# Experiment Tracking with MLflow

## 1.Introduction

This notebook focuses on tracking and managing fine-tuning experiments using MLflow.

The objective is to log model parameters, evaluation metrics, and trained model artifacts for multiple configurations, enabling reproducibility, comparison, and selection of the best-performing model.

This step represents the transition from experimentation to a more structured and production-oriented workflow.

Why MLflow?

MLflow is used to track machine learning experiments in a structured way. It allows logging parameters, metrics, and model artifacts, enabling easy comparison between runs and improving reproducibility.

This is especially useful when multiple model configurations are evaluated, as it provides a centralized way to analyze and select the best-performing model.


## 2. Environment Setup

This section imports the required libraries and defines the main project paths.


In [ ]:
# Import Path to handle file and folder paths in a clean and operating-system-independent way.
from pathlib import Path

# Import json to save configuration and model metadata as structured artifact files.
import json

# Import time to measure evaluation time for each model run.
import time

# Import pandas to load and inspect the test dataset.
import pandas as pd

# Import numpy to work with arrays and prediction outputs.
import numpy as np

# Import torch to run Hugging Face models locally.
import torch

# Import MLflow to track experiments, metrics, parameters, and artifacts.
import mlflow

# Import the Hugging Face tokenizer loader.
from transformers import AutoTokenizer

# Import the Hugging Face sequence classification model loader.
from transformers import AutoModelForSequenceClassification

# Import sklearn metrics for model evaluation.
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# Import matplotlib to create artifact plots.
import matplotlib.pyplot as plt

# Define the project root assuming this notebook is executed from the notebooks folder.
PROJECT_ROOT = Path("..").resolve()

# Define the folder where trained models are stored.
MODELS_DIR = PROJECT_ROOT / "models"

# Define the folder where data splits are stored.
DATA_SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

# Define the path to the test dataset.
TEST_DATA_PATH = DATA_SPLITS_DIR / "test.csv"

# Define the folder where generated reports and artifacts will be saved.
REPORTS_DIR = PROJECT_ROOT / "reports" / "mlflow_artifacts"

# Create the reports folder if it does not exist.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Print the resolved project root to confirm the notebook is using the expected location.
print(f"Project root: {PROJECT_ROOT}")

# Print the expected test dataset path.
print(f"Test data path: {TEST_DATA_PATH}")

# Print the expected models directory.
print(f"Models directory: {MODELS_DIR}")

## 3. Define Model Runs

Each run points to one locally saved Hugging Face model folder.

The run names are intentionally generic and professional. They do not expose internal planning labels.


In [ ]:
# Define the local model folders for each experiment run.
MODEL_RUNS = {
    "run_01_baseline_finetuning": MODELS_DIR / "run_01",
    "run_02_extended_finetuning": MODELS_DIR / "run_02",
    "run_03_optimized_configuration": MODELS_DIR / "run_03",
}

# Define the required files expected inside each Hugging Face model folder.
REQUIRED_MODEL_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "training_args.bin",
]

# Loop through each model run to verify that the expected files exist.
for run_name, model_path in MODEL_RUNS.items():

    # Print the current run being checked.
    print(f"\nChecking {run_name}")

    # Print the model folder path.
    print(f"Path: {model_path}")

    # Check if the model folder exists.
    if not model_path.exists():
        print("Status: Folder not found")
        continue

    # Check each required file inside the model folder.
    for file_name in REQUIRED_MODEL_FILES:

        # Build the full path to the required file.
        file_path = model_path / file_name

        # Print whether the file exists or is missing.
        print(f"{file_name}: {'OK' if file_path.exists() else 'Missing'}")

## 4. Load Test Dataset

This section loads the test split used to evaluate the trained models.

Before running the full notebook, confirm that the text and label column names match your dataset.


In [ ]:
# Load the test dataset from the project data folder.
test_df = pd.read_csv(TEST_DATA_PATH)

# Display the first rows to inspect the dataset structure.
display(test_df.head())

# Print the available column names to confirm which columns contain text and labels.
print("Columns:", list(test_df.columns))

# Define the text column used as model input.
# Update this value if your dataset uses a different column name.
TEXT_COLUMN = "text"

# Define the label column used as the target.
# Update this value if your dataset uses a different column name.
LABEL_COLUMN = "label"

# Validate that the text column exists in the test dataset.
assert TEXT_COLUMN in test_df.columns, f"Missing text column: {TEXT_COLUMN}"

# Validate that the label column exists in the test dataset.
assert LABEL_COLUMN in test_df.columns, f"Missing label column: {LABEL_COLUMN}"

# Remove rows with missing text or label values.
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)

# Convert text values to strings to avoid tokenizer errors.
test_texts = test_df[TEXT_COLUMN].astype(str).tolist()

# Convert labels to integers for metric calculation.
test_labels = test_df[LABEL_COLUMN].astype(int).tolist()

# Print the number of test examples.
print(f"Number of test examples: {len(test_texts)}")

# Print the label distribution in the test set.
print(test_df[LABEL_COLUMN].value_counts().sort_index())

## 5. Define Evaluation Function

This function loads a saved Hugging Face model, runs inference on the test set, calculates metrics, and creates artifacts.


In [ ]:
# Define a function to evaluate one Hugging Face model folder.
def evaluate_model(run_name, model_path, texts, labels, batch_size=16, max_length=128):

    # Record the evaluation start time.
    start_time = time.time()

    # Load the tokenizer from the saved model folder.
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Load the sequence classification model from the saved model folder.
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # Move the model to CPU for local evaluation.
    model.to("cpu")

    # Set the model to evaluation mode.
    model.eval()

    # Create an empty list to store predicted labels.
    all_predictions = []

    # Disable gradient calculation because this is evaluation, not training.
    with torch.no_grad():

        # Iterate over the test texts in batches.
        for start_idx in range(0, len(texts), batch_size):

            # Select one batch of texts.
            batch_texts = texts[start_idx:start_idx + batch_size]

            # Tokenize the batch of texts.
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            # Run the model and obtain logits.
            outputs = model(**inputs)

            # Select the class with the highest logit for each example.
            predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

            # Add the batch predictions to the full prediction list.
            all_predictions.extend(predictions)

    # Convert predictions to a numpy array.
    all_predictions = np.array(all_predictions)

    # Convert labels to a numpy array.
    labels_array = np.array(labels)

    # Calculate accuracy.
    accuracy = accuracy_score(labels_array, all_predictions)

    # Calculate weighted precision, recall, and F1 score.
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_array,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    # Calculate the total evaluation time.
    evaluation_time = time.time() - start_time

    # Create a classification report as a text dictionary.
    report_dict = classification_report(
        labels_array,
        all_predictions,
        zero_division=0,
        output_dict=True
    )

    # Create a classification report as readable text.
    report_text = classification_report(
        labels_array,
        all_predictions,
        zero_division=0
    )

    # Create a confusion matrix.
    cm = confusion_matrix(labels_array, all_predictions)

    # Create the output folder for this run artifacts.
    run_artifact_dir = REPORTS_DIR / run_name

    # Create the artifact folder if it does not exist.
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    # Define the classification report artifact path.
    report_path = run_artifact_dir / "classification_report.txt"

    # Save the classification report as a text artifact.
    report_path.write_text(report_text, encoding="utf-8")

    # Define the metrics artifact path.
    metrics_path = run_artifact_dir / "metrics.json"

    # Store metrics in a dictionary.
    metrics = {
        "accuracy": float(accuracy),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
        "f1_weighted": float(f1),
        "evaluation_time_seconds": float(evaluation_time),
    }

    # Save metrics as a JSON artifact.
    metrics_path.write_text(json.dumps(metrics, indent=4), encoding="utf-8")

    # Define the confusion matrix plot path.
    cm_path = run_artifact_dir / "confusion_matrix.png"

    # Create a confusion matrix plot.
    plt.figure(figsize=(6, 5))

    # Display the confusion matrix values as an image.
    plt.imshow(cm)

    # Add a title to the plot.
    plt.title(f"Confusion Matrix - {run_name}")

    # Add the x-axis label.
    plt.xlabel("Predicted label")

    # Add the y-axis label.
    plt.ylabel("True label")

    # Add a colorbar to help interpret the scale.
    plt.colorbar()

    # Write the numeric values inside each confusion matrix cell.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    # Adjust the layout to avoid clipped labels.
    plt.tight_layout()

    # Save the confusion matrix plot.
    plt.savefig(cm_path)

    # Close the plot to avoid duplicate display.
    plt.close()

    # Return the metrics and artifact paths for MLflow logging.
    return {
        "metrics": metrics,
        "report_path": report_path,
        "metrics_path": metrics_path,
        "confusion_matrix_path": cm_path,
        "report_dict": report_dict,
    }

## 6. Configure MLflow Experiment

This section creates or selects the MLflow experiment used to track all runs.


In [ ]:
# Set the local MLflow tracking directory inside the project.
mlflow.set_tracking_uri(f"file:{PROJECT_ROOT / 'mlruns'}")

# Set a clear experiment name for the project.
mlflow.set_experiment("it_ticket_classification_experiment_tracking")

# Print the active MLflow tracking URI.
print("MLflow tracking URI:", mlflow.get_tracking_uri())

# Print the experiment name being used.
print("MLflow experiment configured successfully.")

## 7. Log Runs to MLflow

This section evaluates each model and logs parameters, metrics, and artifacts to MLflow.


In [ ]:
# Create an empty list to store run summaries.
run_summaries = []

# Loop through each trained model run.
for run_name, model_path in MODEL_RUNS.items():

    # Skip the run if the model folder does not exist.
    if not model_path.exists():
        print(f"Skipping {run_name}: model folder not found.")
        continue

    # Start a new MLflow run using the professional run name.
    with mlflow.start_run(run_name=run_name):

        # Log the model name or base architecture as a parameter.
        mlflow.log_param("base_model", "distilbert-base-uncased")

        # Log the model framework as a parameter.
        mlflow.log_param("framework", "Hugging Face Transformers")

        # Log the task type as a parameter.
        mlflow.log_param("task", "text_classification")

        # Log the local model path as a parameter.
        mlflow.log_param("model_path", str(model_path))

        # Log the test dataset path as a parameter.
        mlflow.log_param("test_data_path", str(TEST_DATA_PATH))

        # Log the number of test examples.
        mlflow.log_param("test_examples", len(test_texts))

        # Log the tokenizer maximum sequence length used during evaluation.
        mlflow.log_param("max_length", 128)

        # Log the batch size used during evaluation.
        mlflow.log_param("eval_batch_size", 16)

        # Evaluate the model and generate artifacts.
        evaluation_result = evaluate_model(
            run_name=run_name,
            model_path=model_path,
            texts=test_texts,
            labels=test_labels,
            batch_size=16,
            max_length=128
        )

        # Log each metric to MLflow.
        for metric_name, metric_value in evaluation_result["metrics"].items():
            mlflow.log_metric(metric_name, metric_value)

        # Log the classification report artifact.
        mlflow.log_artifact(str(evaluation_result["report_path"]))

        # Log the metrics JSON artifact.
        mlflow.log_artifact(str(evaluation_result["metrics_path"]))

        # Log the confusion matrix image artifact.
        mlflow.log_artifact(str(evaluation_result["confusion_matrix_path"]))

        # Log the full Hugging Face model folder as an artifact.
        mlflow.log_artifacts(str(model_path), artifact_path="huggingface_model")

        # Get the active MLflow run ID.
        active_run_id = mlflow.active_run().info.run_id

        # Store a summary of the run for comparison.
        run_summaries.append({
            "run_name": run_name,
            "run_id": active_run_id,
            **evaluation_result["metrics"]
        })

        # Print a completion message for the current run.
        print(f"Logged {run_name} successfully.")

## 8. Compare Runs and Select Best Model

The best model is selected using weighted F1 score because this metric balances precision and recall across classes.


In [ ]:
# Convert the run summaries into a DataFrame.
results_df = pd.DataFrame(run_summaries)

# Sort runs by weighted F1 score in descending order.
results_df = results_df.sort_values(by="f1_weighted", ascending=False).reset_index(drop=True)

# Display the run comparison table.
display(results_df)

# Select the best run from the first row after sorting.
best_run = results_df.iloc[0].to_dict()

# Print the selected best run.
print("Best run selected:")
print(best_run)

## 9. Register Best Model Metadata

This section creates a lightweight model registry artifact.

For a local portfolio project, this is enough to show model selection and registration logic clearly.


In [ ]:
# Define the path where the best model metadata will be stored.
best_model_metadata_path = REPORTS_DIR / "best_model_metadata.json"

# Create a dictionary with best model information.
best_model_metadata = {
    "registered_model_name": "it_ticket_classifier_best_model",
    "selection_metric": "f1_weighted",
    "best_run_name": best_run["run_name"],
    "best_run_id": best_run["run_id"],
    "accuracy": best_run["accuracy"],
    "precision_weighted": best_run["precision_weighted"],
    "recall_weighted": best_run["recall_weighted"],
    "f1_weighted": best_run["f1_weighted"],
    "evaluation_time_seconds": best_run["evaluation_time_seconds"],
}

# Save the best model metadata as a JSON file.
best_model_metadata_path.write_text(json.dumps(best_model_metadata, indent=4), encoding="utf-8")

# Start a final MLflow run to log the best model registration metadata.
with mlflow.start_run(run_name="registered_best_model_metadata"):

    # Log the name of the selected best model.
    mlflow.log_param("registered_model_name", best_model_metadata["registered_model_name"])

    # Log the metric used to select the best model.
    mlflow.log_param("selection_metric", best_model_metadata["selection_metric"])

    # Log the best run name.
    mlflow.log_param("best_run_name", best_model_metadata["best_run_name"])

    # Log the best run ID.
    mlflow.log_param("best_run_id", best_model_metadata["best_run_id"])

    # Log the best weighted F1 score.
    mlflow.log_metric("best_f1_weighted", best_model_metadata["f1_weighted"])

    # Log the best model metadata artifact.
    mlflow.log_artifact(str(best_model_metadata_path))

    # Print confirmation.
    print("Best model metadata registered in MLflow.")

## 10. How to Open the MLflow UI

Run this command from the project root in your terminal:

```bash
mlflow ui
```

Then open:

```text
http://127.0.0.1:5000
```



## 11. ## Conclusion

In this stage, MLflow was successfully integrated to track and manage multiple fine-tuning experiments. Each run captured key information, including model parameters, evaluation metrics, and trained model artifacts, enabling a structured and reproducible experimentation workflow.

This approach allows for efficient comparison between different configurations and supports a data-driven model selection process. The best-performing model was identified based on macro F1-score, ensuring balanced performance across all classes and strong generalization.

It is important to note that while this notebook focuses on the technical implementation of experiment tracking, the visual exploration of MLflow runs (including comparisons, metrics, and selected models) will be presented in the project README. This separation ensures that the notebook remains focused on reproducibility and methodology, while the README provides a clear and concise overview for presentation purposes.

Overall, this stage establishes a solid foundation for experiment tracking, model versioning, and future deployment workflows within a production-oriented machine learning pipeline.
